# Week 2 — Dataset Analysis
Run the cells from top to bottom. **Part 1** loads the five datasets and defines the table style that every later cell uses.

## Part 1 · Load the data

In [ ]:
# 1.1 Imports and loading the five CSV files
import numpy as np
import pandas as pd
import warnings
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

FILES = {
    'Boston':       'data_boston.csv',
    'Census':       'data_census.csv',
    'Coffee Shop':  'data_coffeeshop.csv',
    'Sales':        'data_sales.csv',
    'Student List': 'data_studentlist.csv',
}
datasets = {name: pd.read_csv('/content/' + file) for name, file in FILES.items()}

# Short names for each dataset (the originals are never modified)
df_boston, df_census, df_coffeeshop, df_sales, df_studentlist = datasets.values()

In [ ]:
# 1.2 Display helpers: one shared look for every table
BLUE, LIGHT_BLUE, GREY = '#1a73e8', '#e8f0fe', '#5f6368'

TABLE_STYLE = [
    {'selector': 'th.col_heading', 'props': f'background:{BLUE}; color:white; font-weight:600; padding:7px 12px; text-align:center;'},
    {'selector': 'th.blank',       'props': f'background:{BLUE};'},
    {'selector': 'th.row_heading', 'props': 'background:#f1f3f4; color:#202124; font-weight:600; padding:6px 12px; text-align:left;'},
    {'selector': 'td',             'props': 'padding:6px 12px; text-align:center; border-bottom:1px solid #e8eaed;'},
    {'selector': 'tr:hover td',    'props': f'background-color:{LIGHT_BLUE};'},
]

def clean_num(v):
    """Tidy floats: 18.0 -> 18, 4.0900 -> 4.09"""
    return f'{v:.4f}'.rstrip('0').rstrip('.')

def pretty(df):
    """Styled copy of df: blue header, tidy numbers, '—' for missing."""
    styler = (df.style
                .set_table_styles(TABLE_STYLE)
                .set_table_attributes('style="border-collapse:collapse; font-size:13px; margin:4px 0 10px;"')
                .format(na_rep='—'))
    float_cols = df.select_dtypes('float').columns
    if len(float_cols):
        styler = styler.format(clean_num, subset=float_cols, na_rep='—')
    return styler

def card(title, subtitle, body):
    """Wrap HTML in a rounded white card with a title."""
    return f'''
    <div style="background:white; color:#202124; border:1px solid #dadce0; border-radius:14px;
                padding:16px 22px; margin:16px 0; font-family:Roboto, Arial, sans-serif;">
      <h2 style="margin:0 0 4px; color:{BLUE};">{title}</h2>
      <div style="color:{GREY}; font-size:13px; margin-bottom:10px;">{subtitle}</div>
      <div style="overflow-x:auto;">{body}</div>
    </div>'''

In [ ]:
# 1.3 Preview: the first 2 rows of each dataset
for name, df in datasets.items():
    subtitle = f'{df.shape[0]:,} rows × {df.shape[1]} columns  ·  first 2 rows'
    display(HTML(card(name, subtitle, pretty(df.head(2)).to_html())))

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,PRICE
0,0.0063,18,2.31,0,0.538,6.575,65.2,4.09,1,296,15.3,396.9,4.98,24
1,0.0273,0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.9,9.14,21.6


,Province Code,City Code,Population,Households,Person's Household,Gender Ratio
0,109,1001,"10,078,850","4,197,478",2.4,0.97
1,109,1188,"155,695","72,882",2.14,0.98


,Year of Start,Current State,Size of Site
0,2008,In,20.8
1,2010,In,212.72


,Date,Region,Item,Unit Price,Units
0,04-Jul-14,East,Pen Set,4.99,62
1,12-Jul-14,East,Binder,1.99,29


,Name,Gender,Age,Grade,Absence,Bloodtype,Height,Weight
0,Jared Diamond,M,23,3,Y,O,165.3,68.2
1,Sarah O'Donnel,F,22,2,N,AB,170.1,53


## Part 2 · Exploratory analysis
One card per dataset: size, missing values, duplicates, column overview and descriptive statistics.

In [ ]:
# 2.0 EDA helpers
RED, LIGHT_RED = '#d93025', '#fce8e6'

def stat_tiles(df):
    """Row of summary tiles: rows, columns, missing cells, duplicates (red when > 0)."""
    rows, cols = df.shape
    missing = int(df.isnull().sum().sum())
    dups = int(df.duplicated().sum())
    tiles = [('Rows', f'{rows:,}', False),
             ('Columns', f'{cols}', False),
             ('Missing cells', f'{missing:,}', missing > 0),
             ('Duplicate rows', f'{dups:,} ({dups / max(rows, 1):.1%})', dups > 0)]
    html = ''
    for label, value, warn in tiles:
        bg, fg = (LIGHT_RED, RED) if warn else (LIGHT_BLUE, '#174ea6')
        html += f'''
        <div style="background:{bg}; border-radius:10px; padding:10px 18px; min-width:120px;">
          <div style="font-size:12px; color:{GREY};">{label}</div>
          <div style="font-size:22px; font-weight:600; color:{fg};">{value}</div>
        </div>'''
    return f'<div style="display:flex; gap:12px; flex-wrap:wrap; margin:6px 0;">{html}</div>'

def section(title, table, empty_msg=''):
    """Small heading followed by a styled table (or a grey note)."""
    body = table.to_html() if table is not None else f'<i style="color:{GREY};">{empty_msg}</i>'
    return f'<h4 style="margin:20px 0 6px; color:#202124;">{title}</h4>{body}'

def eda_card(name):
    """Full EDA card for one dataset."""
    df = datasets[name]
    info = pd.DataFrame({
        'Data Type':      df.dtypes.astype(str),
        'Non-Null Count': df.count(),
        'Null Count':     df.isnull().sum(),
        'Null %':         df.isnull().mean() * 100,
        'Unique Values':  df.nunique(),
    })
    info_tbl = (pretty(info)
                .format('{:.2f}%', subset=['Null %'])
                .bar(subset=['Null %'], color='#f6aea9', vmin=0, vmax=100)
                .apply(lambda col: [f'color:{RED}; font-weight:600' if v > 0 else '' for v in col],
                       subset=['Null Count']))
    num_df = df.select_dtypes(include=np.number)
    cat_df = df.select_dtypes(exclude=np.number)
    num_tbl = pretty(num_df.describe().T) if num_df.shape[1] else None
    cat_tbl = pretty(cat_df.describe(include='all').T) if cat_df.shape[1] else None
    body = (stat_tiles(df)
            + section('Columns overview', info_tbl)
            + section('Numerical statistics', num_tbl, 'No numerical columns found.')
            + section('Categorical statistics', cat_tbl, 'No categorical columns found.'))
    display(HTML(card(name, 'Data quality & descriptive statistics', body)))

In [ ]:
# 2.1 EDA · Boston
eda_card('Boston')

,Data Type,Non-Null Count,Null Count,Null %,Unique Values
CRIM,float64,506,0,0.00%,504
ZN,float64,506,0,0.00%,26
INDUS,float64,506,0,0.00%,76
CHAS,int64,506,0,0.00%,2
NOX,float64,506,0,0.00%,81
RM,float64,506,0,0.00%,446
AGE,float64,506,0,0.00%,356
DIS,float64,506,0,0.00%,412
RAD,int64,506,0,0.00%,9
TAX,int64,506,0,0.00%,66


In [ ]:
# 2.2 EDA · Census
eda_card('Census')

In [ ]:
# 2.3 EDA · Coffee Shop
eda_card('Coffee Shop')

In [ ]:
# 2.4 EDA · Sales
eda_card('Sales')

In [ ]:
# 2.5 EDA · Student List
eda_card('Student List')

## Part 3 · Audit: loading & reproducibility
Each CSV is re-read from disk and every statistic is recomputed with NumPy, to prove the results above are correct.

In [ ]:
# 3.1 Audit function (read-only)
import hashlib, csv

def audit_file(name, file):
    """Re-read one CSV and check encoding, delimiter, size, integrity and statistics."""
    raw = open('/content/' + file, 'rb').read()
    try:
        raw.decode('utf-8'); utf8 = True
    except UnicodeDecodeError:
        utf8 = False
    delim = csv.Sniffer().sniff(raw[:4000].decode('utf-8', 'replace'), delimiters=',;\t|').delimiter
    fresh, mem = pd.read_csv('/content/' + file), datasets[name]

    # Recompute describe() with NumPy and keep the largest difference
    diffs = [0.0]
    for col in fresh.select_dtypes(include=np.number).columns:
        x = fresh[col].dropna().to_numpy(dtype=float)
        mine = [len(x), x.mean(), x.std(ddof=1), x.min(), *np.percentile(x, [25, 50, 75]), x.max()]
        diffs.append(float(np.max(np.abs(mem[col].describe().to_numpy() - np.array(mine)))))

    # Text columns whose values are really numbers (e.g. ' 10,078,850 ')
    text_numbers = [c for c in fresh.select_dtypes(exclude=np.number)
                    if pd.to_numeric(fresh[c].dropna().astype(str).str.strip().str.replace(',', ''),
                                     errors='coerce').notna().all()]
    return {'MD5 (first 8)': hashlib.md5(raw).hexdigest()[:8], 'UTF-8': utf8, 'Delimiter': repr(delim),
            'Lines incl. header': raw.count(b'\n'), 'Rows': fresh.shape[0], 'Cols': fresh.shape[1],
            'Memory = file': mem.equals(fresh), 'Missing cells': int(fresh.isnull().sum().sum()),
            'Duplicate rows': int(fresh.duplicated().sum()),
            'Text cols that are numbers': ', '.join(text_numbers) or '—',
            'Max |describe − NumPy|': f'{max(diffs):.1e}'}

In [ ]:
# 3.2 Run the audit on all five files
audit = pd.DataFrame.from_dict({name: audit_file(name, file) for name, file in FILES.items()}, orient='index')
display(HTML(card('Audit · loading & reproducibility',
                  'Each CSV re-read from disk and every statistic recomputed with NumPy',
                  pretty(audit).to_html())))

,MD5 (first 8),UTF-8,Delimiter,Lines incl. header,Rows,Cols,Memory = file,Missing cells,Duplicate rows,Text cols that are numbers,Max |describe − NumPy|
Boston,1619a15a,True,"','",507,506,14,True,0,0,—,2.3e-13
Census,157a04fc,True,"','",282,281,6,True,0,1,"Population, Households",1.4e-14
Coffee Shop,cdda0062,True,"','",43183,43182,3,True,57,11747,—,2.4e-12
Sales,2fb09d71,True,"','",44,43,5,True,0,0,—,2.8e-14
Student List,3de9de0d,True,"','",18,17,8,True,0,0,—,8.9e-16


## Part 4 · Corrective EDA
Fixes what the audit found: correct data types, mode, frequencies, outliers, correlations, group comparisons and preprocessing notes.
Everything works on **copies** — the original DataFrames are never changed.

In [ ]:
# 4.1 Typed working copies (the preprocessing the audit found missing)
w_boston = df_boston.copy()
w_boston['CHAS'] = w_boston['CHAS'].astype('category')      # 0/1 dummy, not a measurement
w_boston['RAD']  = w_boston['RAD'].astype('category')       # accessibility index

w_census = df_census.copy()
for col in ['Population', 'Households']:                    # stored as text like ' 10,078,850 '
    w_census[col] = pd.to_numeric(w_census[col].str.strip().str.replace(',', '', regex=False))
w_census['Is Province Total'] = w_census['City Code'].eq(1001)   # City Code 1001 = province total row
w_census['Province Code'] = w_census['Province Code'].astype('category')
w_census['City Code']     = w_census['City Code'].astype('category')

w_coffee = df_coffeeshop.copy()
w_coffee['Year of Start'] = w_coffee['Year of Start'].astype('Int64')   # float only because of NaN
w_coffee['Current State'] = w_coffee['Current State'].astype('category')

w_sales = df_sales.copy()
w_sales['Date']    = pd.to_datetime(w_sales['Date'], format='%d-%b-%y')
w_sales['Region']  = w_sales['Region'].astype('category')
w_sales['Item']    = w_sales['Item'].astype('category')
w_sales['Revenue'] = w_sales['Unit Price'] * w_sales['Units']   # derived column (not in the file)

w_students = df_studentlist.copy()
for col in ['Gender', 'Absence', 'Bloodtype']:
    w_students[col] = w_students[col].astype('category')
w_students['Grade'] = pd.Categorical(w_students['Grade'], ordered=True)

work = {'Boston': w_boston, 'Census': w_census, 'Coffee Shop': w_coffee,
        'Sales': w_sales, 'Student List': w_students}

In [ ]:
# 4.2 Variable roles: how each column should be analysed
ROLES = {
    'Boston':       {'CHAS': 'Binary (0/1)', 'RAD': 'Ordinal index',
                     **{k: 'Continuous' for k in ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS',
                                                  'TAX', 'PTRATIO', 'B', 'LSTAT', 'PRICE']}},
    'Census':       {'Province Code': 'Identifier (code)', 'City Code': 'Identifier (code)',
                     'Population': 'Count', 'Households': 'Count',
                     "Person's Household": 'Ratio', 'Gender Ratio': 'Ratio'},
    'Coffee Shop':  {'Year of Start': 'Date-related (year)', 'Current State': 'Binary (In/Out)',
                     'Size of Site': 'Continuous'},
    'Sales':        {'Date': 'Date', 'Region': 'Nominal', 'Item': 'Nominal',
                     'Unit Price': 'Continuous', 'Units': 'Count'},
    'Student List': {'Name': 'Identifier (text)', 'Gender': 'Binary (M/F)', 'Age': 'Discrete',
                     'Grade': 'Ordinal', 'Absence': 'Binary (Y/N)', 'Bloodtype': 'Nominal',
                     'Height': 'Continuous', 'Weight': 'Continuous'},
}
MEASURES   = ('Continuous', 'Count', 'Ratio', 'Discrete', 'Date-related (year)')
CATEGORIES = ('Binary', 'Ordinal', 'Nominal')

def roles_table(name):
    """Original dtype vs working dtype vs role, one row per column."""
    orig, w = datasets[name], work[name]
    return pd.DataFrame({'Original dtype': orig.dtypes.astype(str),
                         'Working dtype':  w[orig.columns].dtypes.astype(str),
                         'Role':           pd.Series(ROLES[name])}).reindex(orig.columns)

In [ ]:
# 4.3 Analysis helpers
def mode_text(x):
    """Mode(s) as text; 'none' when every value appears equally often."""
    m = x.mode()
    if len(m) == x.nunique() and len(m) > 1:
        return 'none (all values tie)'
    txt = ', '.join(clean_num(float(v)) for v in m.head(3))
    return txt + (f' (+{len(m) - 3} more)' if len(m) > 3 else '')

def full_stats(df, cols):
    """Count, mean, median, mode, std, min, Q1, Q3, max, skew and IQR outliers."""
    rows = {}
    for col in cols:
        x = df[col].dropna().astype(float)
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        iqr = q3 - q1
        rows[col] = {'Count': len(x), 'Mean': x.mean(), 'Median': x.median(), 'Mode': mode_text(x),
                     'Std': x.std(), 'Min': x.min(), 'Q1': q1, 'Q3': q3, 'Max': x.max(),
                     'Skew': round(x.skew(), 2),
                     'IQR outliers': int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum())}
    return pd.DataFrame.from_dict(rows, orient='index')

def freq_tables(df, cols):
    """Side-by-side frequency tables (count and %)."""
    html = ''
    for col in cols:
        vc = df[col].value_counts(dropna=False)
        t = pd.DataFrame({'Count': vc, '%': vc / len(df) * 100})
        t.index = t.index.astype(str).rename(None)
        html += f'<div>{pretty(t).format("{:.1f}%", subset=["%"]).set_caption(f"<b>{col}</b>").to_html()}</div>'
    return f'<div style="display:flex; gap:24px; flex-wrap:wrap; align-items:flex-start;">{html}</div>'

def corr_html(df, method='pearson'):
    """Correlation matrix coloured from blue (-1) to red (+1)."""
    return (pretty(df.corr(method=method)).format('{:.3f}')
            .background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).to_html())

def h4(text):
    return f'<h4 style="margin:20px 0 6px; color:#202124;">{text}</h4>'

def bullets(items):
    return '<ul style="margin:4px 0 8px 18px; line-height:1.55;">' + ''.join(f'<li>{i}</li>' for i in items) + '</ul>'

def corrective_card(name, quality, relations, prep, rows=None, note=''):
    """Render one corrective-EDA card. `rows` optionally limits which rows feed the statistics."""
    w = work[name]
    measures = [k for k, v in ROLES[name].items() if v in MEASURES]
    cats = [k for k, v in ROLES[name].items() if v.startswith(CATEGORIES)]
    data = w if rows is None else w[rows]
    body = (h4('Variable roles and types') + pretty(roles_table(name)).to_html()
            + h4('Data-quality findings') + bullets(quality)
            + h4('Descriptive statistics' + note) + pretty(full_stats(data, measures)).to_html()
            + (h4('Frequencies') + freq_tables(w, cats) if cats else '')
            + h4('Relationships (associations only — not causal)') + relations
            + h4('Preprocessing requirements (not applied to the originals)') + bullets(prep))
    display(HTML(card(name, 'Corrective EDA · audit follow-up', body)))

In [ ]:
# 4.4 Corrective EDA · Boston
b = w_boston
cont = [k for k, v in ROLES['Boston'].items() if v == 'Continuous']

corr_price = pd.DataFrame({'Pearson r':  b[cont].corr()['PRICE'],
                           'Spearman ρ': b[cont].corr(method='spearman')['PRICE']}
                          ).drop('PRICE').sort_values('Pearson r')
pairs = (b[cont].corr().where(np.triu(np.ones((len(cont),) * 2, bool), 1)).stack()
         .sort_values(key=abs, ascending=False).head(6))
pairs_df = pd.DataFrame({'Variable A': [p[0] for p in pairs.index],
                         'Variable B': [p[1] for p in pairs.index],
                         'Pearson r':  pairs.values})
price_by_chas = b.groupby('CHAS', observed=True)['PRICE'].agg(['count', 'mean', 'median'])
n_capped = int((b['PRICE'] == 50).sum())

corrective_card(
    'Boston',
    quality=[f'No missing values; {int(df_boston.duplicated().sum())} duplicate rows.',
             f'PRICE has {n_capped} values exactly at the maximum 50 → likely capped (censored) at 50.',
             f'ZN is 0 in {int((b["ZN"] == 0).sum())} of {len(b)} rows; CRIM is highly right-skewed (skew {b["CRIM"].skew():.2f}).',
             'CHAS (0/1 dummy) and RAD (index) are stored as integers but are categorical — excluded from correlations.'],
    relations=(h4('Correlation of each continuous variable with PRICE')
               + pretty(corr_price).format('{:.3f}').background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).to_html()
               + h4('Strongest pairwise correlations (|r|)')
               + pretty(pairs_df).format('{:.3f}', subset=['Pearson r']).hide(axis='index').to_html()
               + h4('PRICE by CHAS (river dummy)') + pretty(price_by_chas).to_html()),
    prep=['Treat CHAS and RAD as categorical.',
          'Consider the PRICE cap at 50 in any modelling.',
          'Consider log-transform or robust methods for skewed CRIM, ZN, B.'])

,Original dtype,Working dtype,Role
CRIM,float64,float64,Continuous
ZN,float64,float64,Continuous
INDUS,float64,float64,Continuous
CHAS,int64,category,Binary (0/1)
NOX,float64,float64,Continuous
RM,float64,float64,Continuous
AGE,float64,float64,Continuous
DIS,float64,float64,Continuous
RAD,int64,category,Ordinal index
TAX,int64,int64,Continuous


In [ ]:
# 4.5 Corrective EDA · Census
c = w_census
totals = c[c['Is Province Total']]                     # one row per province (City Code 1001)
cities = c[~c['Is Province Total']]
excess = (cities.groupby('Province Code', observed=True)['Population'].sum()
          - totals.drop_duplicates().groupby('Province Code', observed=True)['Population'].sum()).fillna(0)
over = [str(p) for p, v in excess.items() if v > 0]   # provinces that also contain sub-total rows
dup = df_census[df_census.duplicated(keep=False)]
ratio_gap = (c['Population'] / c['Households'] - c["Person's Household"]).abs().max()
provinces = (totals.drop_duplicates().set_index('Province Code')
             [['Population', 'Households', "Person's Household", 'Gender Ratio']]
             .sort_values('Population', ascending=False))

corrective_card(
    'Census',
    quality=['Population and Households are stored as <b>text</b> with spaces and thousands separators '
             f'(e.g. "{df_census["Population"].iloc[0]}") → the earlier analysis treated them as categorical. Converted in the copy.',
             f'{len(totals)} rows have City Code 1001 = <b>province totals</b> mixed with city rows '
             f'(totals sum to {int(totals.drop_duplicates()["Population"].sum()):,}; all rows sum to {int(c["Population"].sum()):,} → double counting).',
             f'City-level rows still exceed their province total in provinces {", ".join(over)} → intermediate sub-totals '
             '(cities with districts) are also present and cannot be separated without a code book.',
             f'{int(df_census.duplicated().sum())} exact duplicate row (Province {dup["Province Code"].iloc[0]}, City Code {dup["City Code"].iloc[0]}).',
             f'City Code is not unique on its own ({int(df_census["City Code"].duplicated().sum())} repeats) → identify cities by (Province Code, City Code).',
             f"Person's Household matches Population ÷ Households (max gap {ratio_gap:.3f}) → internally consistent."],
    relations=(h4('Province comparison (province-total rows, City Code 1001)') + pretty(provinces).to_html()
               + h4('Spearman correlations — city-level rows (province totals excluded)')
               + corr_html(cities[['Population', 'Households', "Person's Household", 'Gender Ratio']], 'spearman')),
    prep=['Convert Population/Households to numbers (strip spaces, remove commas).',
          'Exclude or flag province-total rows (City Code 1001) before summing or averaging.',
          'Obtain a code book to separate the remaining sub-total rows.',
          'Treat Province/City Code as identifiers, not numbers.',
          'Decide whether to drop the duplicate row.'],
    rows=~c['Is Province Total'],
    note=' — city-level rows only (province totals excluded)')

In [ ]:
# 4.6 Corrective EDA · Coffee Shop
cs = w_coffee
size = cs['Size of Site']
n_zero, top2 = int((size == 0).sum()), size.nlargest(2).tolist()
by_state = cs[size > 0].groupby('Current State', observed=True).agg(
    Sites=('Size of Site', 'size'), Mean_Size=('Size of Site', 'mean'),
    Median_Size=('Size of Site', 'median'), Median_Start_Year=('Year of Start', 'median'))
rho = cs[['Year of Start', 'Size of Site']].astype(float).corr(method='spearman').iloc[0, 1]

corrective_card(
    'Coffee Shop',
    quality=[f'{int(cs.isnull().all(axis=1).sum())} rows are completely empty (all 3 columns missing).',
             f'{int(df_coffeeshop.duplicated().sum()):,} exact duplicate rows ({df_coffeeshop.duplicated().mean():.1%}). '
             'There is no ID column, so these may be different shops with identical values — not removed.',
             f'{n_zero:,} sites have Size of Site = 0 (likely missing/invalid).',
             f'Maximum size {top2[0]:,} vs next largest {top2[1]:,} → probable entry error; size skew {size.skew():.1f}.',
             'Year of Start is a float only because of NaN; it is a date-related (year) variable.'],
    relations=(h4('By Current State (sites with size &gt; 0)') + pretty(by_state).to_html()
               + f'<p>Spearman ρ (Year of Start vs Size of Site) = <b>{rho:.3f}</b></p>'),
    prep=['Drop or flag the fully empty rows.',
          'Treat Size = 0 as missing or investigate.',
          f'Investigate the {top2[0]:,.0f} outlier.',
          'Use medians / robust statistics for Size of Site.',
          'Store Year of Start as integer (nullable).'])

In [ ]:
# 4.7 Corrective EDA · Sales
s = w_sales
summary = dict(Orders=('Units', 'size'), Units=('Units', 'sum'),
               Revenue=('Revenue', 'sum'), Avg_Unit_Price=('Unit Price', 'mean'))
by_region = s.groupby('Region', observed=True).agg(**summary)
by_item   = s.groupby('Item', observed=True).agg(**summary)
monthly = s.set_index('Date')['Revenue'].resample('MS').sum().to_frame()
monthly.index = monthly.index.strftime('%Y-%m')
r_price_units = s['Unit Price'].corr(s['Units'])
prices_per_item = s.groupby('Item', observed=True)['Unit Price'].nunique()

corrective_card(
    'Sales',
    quality=[f'Date was stored as text; parsed with format dd-Mon-yy. Range {s["Date"].min():%Y-%m-%d} to {s["Date"].max():%Y-%m-%d}.',
             f'Unit Price varies within the same item ({", ".join(f"{k}: {v}" for k, v in prices_per_item.items())} distinct prices) — likely product variants.',
             f'No missing values or duplicates; small sample ({len(s)} rows).',
             'Revenue is not in the file; it is derived here as Unit Price × Units.'],
    relations=(h4('By Region') + pretty(by_region).to_html()
               + h4('By Item') + pretty(by_item).to_html()
               + h4('Revenue by month') + pretty(monthly).to_html()
               + f'<p>Pearson r (Unit Price vs Units) = <b>{r_price_units:.3f}</b></p>'),
    prep=['Convert Date to datetime.', 'Derive Revenue when totals are needed.'])

In [ ]:
# 4.8 Corrective EDA · Student List
st = w_students
by_gender = st.groupby('Gender', observed=True).agg(Students=('Name', 'size'), Age=('Age', 'mean'),
                                                    Height=('Height', 'mean'), Weight=('Weight', 'mean'))
absence_by_grade = pd.crosstab(st['Grade'], st['Absence'])
r_hw = st['Height'].corr(st['Weight'])

corrective_card(
    'Student List',
    quality=[f'No missing values or duplicates; very small sample ({len(st)} students) — statistics are fragile.',
             'Name is a unique identifier; Grade is ordinal; Gender and Absence are binary.',
             f'Weight has no meaningful mode (all {st["Weight"].nunique()} values are different).'],
    relations=(h4('By Gender') + pretty(by_gender).to_html()
               + h4('Absence by Grade') + pretty(absence_by_grade).to_html()
               + h4('Correlations (Age, Height, Weight)') + corr_html(st[['Age', 'Height', 'Weight']])
               + f'<p>Height–Weight Pearson r = <b>{r_hw:.3f}</b>.</p>'),
    prep=['Treat Grade as ordinal and Gender/Absence/Bloodtype as categorical.'])

In [ ]:
# 4.9 Integrity check: the original DataFrames still match the files on disk
integrity = pd.DataFrame.from_dict(
    {name: {'Rows': datasets[name].shape[0], 'Columns': datasets[name].shape[1],
            'Identical to file': datasets[name].equals(pd.read_csv('/content/' + file))}
     for name, file in FILES.items()},
    orient='index')
display(HTML(card('Integrity check', 'Original DataFrames compared with a fresh read of each CSV',
                  pretty(integrity).to_html())))

,Rows,Columns,Identical to file
Boston,506,14,True
Census,281,6,True
Coffee Shop,43182,3,True
Sales,43,5,True
Student List,17,8,True
